# Phase 5 — Error Analysis and Model Iteration
**Goal:** Understand failure modes and make targeted improvements.

### Milestones
1. Extract all misclassified papers from the test set
2. Identify top 10 most confused category pairs
3. Hypothesize root cause per confused pair
4. Decide + implement at least one targeted fix
5. Retrain and confirm improvement
6. Re-evaluate on full test set
7. Document what worked and what didn't

**Make sure you ran Phase 4 cells 1–6 (mount drive, install, setup, load data) before running this notebook.**

In [ ]:
# ── CELL 1: Mount Drive & setup (skip if already done) ───────────────────────
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os
EXTRACT_DIR = '/content/PaperChaseAI-main'
ZIP_PATH    = '/content/drive/MyDrive/PaperChaseAI-main.zip'

if not os.path.exists(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')
    print('Extracted.')
else:
    print('Already extracted.')

os.chdir(EXTRACT_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# ── CELL 2: Installs ──────────────────────────────────────────────────────────
!pip install -q transformers==4.40.0 scikit-learn pandas torch seaborn joblib tqdm
print('Done.')

In [ ]:
# ── CELL 3: Imports & paths ───────────────────────────────────────────────────
import os, json, warnings, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from tqdm.notebook import tqdm
from collections import Counter

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
warnings.filterwarnings('ignore')

OUTPUT_DIR    = 'outputs'
TRAIN_CSV     = f'{OUTPUT_DIR}/train.csv'
VAL_CSV       = f'{OUTPUT_DIR}/val.csv'
TEST_CSV      = f'{OUTPUT_DIR}/test.csv'
LE_CAT_PATH   = f'{OUTPUT_DIR}/le_main_category.pkl'
LE_SUB_PATH   = f'{OUTPUT_DIR}/le_sub_category.pkl'
CAT_CKPT_DIR  = f'{OUTPUT_DIR}/scibert_main_category/best_model'
SUB_CKPT_DIR  = f'{OUTPUT_DIR}/scibert_sub_category/best_model'
HIER_MAP_PATH = f'{OUTPUT_DIR}/phase4_hierarchy_map.json'
P5_DIR        = f'{OUTPUT_DIR}/phase5'
os.makedirs(P5_DIR, exist_ok=True)

device = (
    torch.device('cuda') if torch.cuda.is_available() else
    torch.device('mps')  if torch.backends.mps.is_available() else
    torch.device('cpu')
)
print(f'Device: {device}')

# ── Load data & encoders ──────────────────────────────────────────────────────
train = pd.read_csv(TRAIN_CSV)
val   = pd.read_csv(VAL_CSV)
test  = pd.read_csv(TEST_CSV)
le_cat = joblib.load(LE_CAT_PATH)
le_sub = joblib.load(LE_SUB_PATH)
num_cat = len(le_cat.classes_)
num_sub = len(le_sub.classes_)

with open(HIER_MAP_PATH) as f:
    hierarchy = json.load(f)

mask_matrix = np.zeros((num_cat, num_sub), dtype=bool)
for cat_name, sub_names in hierarchy.items():
    c = le_cat.transform([cat_name])[0]
    for s_name in sub_names:
        if s_name in le_sub.classes_:
            mask_matrix[c, le_sub.transform([s_name])[0]] = True

print(f'Train: {len(train):,}  Val: {len(val):,}  Test: {len(test):,}')
print(f'Categories: {num_cat}  Sub-categories: {num_sub}')

In [ ]:
# ── CELL 4: Dataset & inference helpers ──────────────────────────────────────
class PaperDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, label_col):
        self.texts   = df['text'].fillna('').tolist()
        self.labels  = df[label_col].tolist()
        self.tok     = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(self.texts[idx], max_length=self.max_len,
                       padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long),
        }

def get_preds_and_logits(model, loader):
    model.eval()
    logits_list, preds_list, labels_list = [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='Inference', leave=False):
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            out  = model(input_ids=ids, attention_mask=mask)
            logits_list.append(out.logits.cpu().numpy())
            preds_list.extend(out.logits.argmax(-1).cpu().numpy())
            labels_list.extend(batch['label'].numpy())
    return np.vstack(logits_list), np.array(preds_list), np.array(labels_list)

def evaluate_loader(model, loader, loss_fn):
    model.eval()
    total_loss, preds_list, labels_list = 0.0, [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            lbls = batch['label'].to(device)
            out  = model(input_ids=ids, attention_mask=mask)
            total_loss += loss_fn(out.logits, lbls).item()
            preds_list.extend(out.logits.argmax(-1).cpu().numpy())
            labels_list.extend(lbls.cpu().numpy())
    return (total_loss / len(loader),
            f1_score(labels_list, preds_list, average='macro', zero_division=0),
            accuracy_score(labels_list, preds_list))

print('Helpers defined.')

In [ ]:
# ── CELL 5: Load both models & run inference on test set ─────────────────────
MAX_LEN     = 256
BATCH_SIZE  = 64
NUM_WORKERS = 2 if device.type == 'cuda' else 0

tokenizer = AutoTokenizer.from_pretrained(SUB_CKPT_DIR)

test_cat_ds = PaperDataset(test, tokenizer, MAX_LEN, 'cat_label')
test_sub_ds = PaperDataset(test, tokenizer, MAX_LEN, 'sub_label')

loader_cat = DataLoader(test_cat_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'))
loader_sub = DataLoader(test_sub_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'))

cat_model = AutoModelForSequenceClassification.from_pretrained(CAT_CKPT_DIR).to(device)
sub_model = AutoModelForSequenceClassification.from_pretrained(SUB_CKPT_DIR).to(device)

print('Running category model...')
cat_logits, cat_preds, cat_true = get_preds_and_logits(cat_model, loader_cat)

print('Running sub-category model...')
sub_logits, sub_preds_unc, sub_true = get_preds_and_logits(sub_model, loader_sub)

# Apply hierarchical constraint
NEG_INF = -1e9
sub_preds_con = np.empty(len(sub_true), dtype=int)
for i, (logit_row, pred_cat) in enumerate(zip(sub_logits, cat_preds)):
    masked = logit_row.copy().astype(float)
    masked[mask_matrix[pred_cat] == 0] = NEG_INF
    sub_preds_con[i] = int(np.argmax(masked))

cat_acc  = accuracy_score(cat_true, cat_preds)
cat_mf1  = f1_score(cat_true, cat_preds, average='macro', zero_division=0)
sub_acc  = accuracy_score(sub_true, sub_preds_con)
sub_mf1  = f1_score(sub_true, sub_preds_con, average='macro', zero_division=0)
joint_acc = np.mean((cat_preds == cat_true) & (sub_preds_con == sub_true))

print(f'\nCategory:   acc={cat_acc:.4f}  macro-F1={cat_mf1:.4f}')
print(f'Sub-cat:    acc={sub_acc:.4f}  macro-F1={sub_mf1:.4f}')
print(f'Joint acc:  {joint_acc:.4f}')

In [ ]:
# ── CELL 6: MILESTONE 1 — Extract all misclassified papers ───────────────────
print('=' * 60)
print('MILESTONE 1 — Misclassified papers')
print('=' * 60)

test_copy = test.copy().reset_index(drop=True)
test_copy['true_cat']      = le_cat.inverse_transform(cat_true)
test_copy['pred_cat']      = le_cat.inverse_transform(cat_preds)
test_copy['true_sub']      = le_sub.inverse_transform(sub_true)
test_copy['pred_sub_con']  = le_sub.inverse_transform(sub_preds_con)
test_copy['cat_correct']   = (cat_preds == cat_true)
test_copy['sub_correct']   = (sub_preds_con == sub_true)
test_copy['joint_correct'] = test_copy['cat_correct'] & test_copy['sub_correct']

cat_errors  = test_copy[~test_copy['cat_correct']]
sub_errors  = test_copy[~test_copy['sub_correct']]
joint_errors = test_copy[~test_copy['joint_correct']]

cat_errors.to_csv(f'{P5_DIR}/cat_errors.csv', index=False)
sub_errors.to_csv(f'{P5_DIR}/sub_errors.csv', index=False)
joint_errors.to_csv(f'{P5_DIR}/joint_errors.csv', index=False)

print(f'  Total test samples:        {len(test_copy):,}')
print(f'  Category errors:           {len(cat_errors):,}  ({len(cat_errors)/len(test_copy)*100:.1f}%)')
print(f'  Sub-category errors:       {len(sub_errors):,}  ({len(sub_errors)/len(test_copy)*100:.1f}%)')
print(f'  Joint errors:              {len(joint_errors):,}  ({len(joint_errors)/len(test_copy)*100:.1f}%)')
print(f'\n  Saved -> {P5_DIR}/')

In [ ]:
# ── CELL 7: MILESTONE 2 — Top 10 most confused category pairs ────────────────
print('=' * 60)
print('MILESTONE 2 — Top 10 confused category pairs')
print('=' * 60)

confused_pairs = Counter()
for tc, pc in zip(test_copy['true_cat'], test_copy['pred_cat']):
    if tc != pc:
        confused_pairs[(tc, pc)] += 1

top10_pairs = confused_pairs.most_common(10)
top10_df = pd.DataFrame([
    {'true_cat': tc, 'pred_cat': pc, 'count': n,
     'pct_of_errors': round(n / len(cat_errors) * 100, 1)}
    for (tc, pc), n in top10_pairs
])
print(top10_df.to_string(index=False))
top10_df.to_csv(f'{P5_DIR}/top10_confused_pairs.csv', index=False)

# Also top 10 confused sub-category pairs
confused_sub_pairs = Counter()
for ts, ps in zip(test_copy['true_sub'], test_copy['pred_sub_con']):
    if ts != ps:
        confused_sub_pairs[(ts, ps)] += 1

top10_sub = pd.DataFrame([
    {'true_sub': ts, 'pred_sub': ps, 'count': n}
    for (ts, ps), n in confused_sub_pairs.most_common(10)
])
print('\nTop 10 confused sub-category pairs:')
print(top10_sub.to_string(index=False))
top10_sub.to_csv(f'{P5_DIR}/top10_confused_sub_pairs.csv', index=False)

# Confusion matrix heatmap
cm = confusion_matrix(cat_true, cat_preds)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_cat.classes_, yticklabels=le_cat.classes_,
            ax=ax, linewidths=0.4, cbar=False, annot_kws={'size': 9})
ax.set_title('Phase 5 — Category Confusion Matrix', fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(f'{P5_DIR}/confusion_matrix_cat.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print(f'\n  Saved -> {P5_DIR}/confusion_matrix_cat.png')

In [ ]:
# ── CELL 8: MILESTONE 3 — Root cause analysis (5–10 examples per pair) ───────
print('=' * 60)
print('MILESTONE 3 — Manual root cause analysis')
print('=' * 60)

N_EXAMPLES = 7  # examples to show per confused pair

root_cause_notes = {}  # fill this in as you read examples

for (true_cat, pred_cat), count in top10_pairs[:5]:  # top 5 pairs
    examples = test_copy[
        (test_copy['true_cat'] == true_cat) &
        (test_copy['pred_cat'] == pred_cat)
    ][['text', 'true_cat', 'pred_cat', 'true_sub', 'pred_sub_con']].head(N_EXAMPLES)

    print(f'\n{"-"*60}')
    print(f'  TRUE: {true_cat}  →  PREDICTED: {pred_cat}  ({count} errors)')
    print(f'{"-"*60}')
    for i, row in examples.iterrows():
        snippet = str(row['text'])[:300].replace('\n', ' ')
        print(f'  [{i}] sub: {row["true_sub"]} → {row["pred_sub_con"]}')
        print(f'       "{snippet}..."\n')

    # Possible root causes — edit these after reading examples
    root_cause_notes[f'{true_cat} -> {pred_cat}'] = {
        'count': count,
        'hypothesis': 'TO FILL: label_noise | topic_overlap | short_abstract | domain_jargon | model_limitation',
        'suggested_fix': 'TO FILL: relabel | oversample | merge_classes | more_epochs | focal_loss'
    }

# Save root cause template
with open(f'{P5_DIR}/root_cause_analysis.json', 'w') as f:
    json.dump(root_cause_notes, f, indent=2)
print(f'\n  Template saved -> {P5_DIR}/root_cause_analysis.json')
print('  ⚠️  Edit the hypothesis and suggested_fix fields after reading the examples above.')

In [ ]:
# ── CELL 9: MILESTONE 4 — Class imbalance analysis & fix decision ─────────────
print('=' * 60)
print('MILESTONE 4 — Imbalance analysis & targeted fix')
print('=' * 60)

# Sub-category support in train set
sub_counts = train['sub_label'].value_counts().sort_index()
sub_names  = le_sub.inverse_transform(sub_counts.index)
support_df = pd.DataFrame({'sub_category': sub_names, 'train_count': sub_counts.values})

# Per-class F1
report = classification_report(
    sub_true, sub_preds_con,
    target_names=le_sub.classes_, zero_division=0, output_dict=True
)
per_class_f1 = pd.DataFrame([
    {'sub_category': k, 'f1': round(v['f1-score'], 4),
     'precision': round(v['precision'], 4), 'recall': round(v['recall'], 4),
     'support': v['support']}
    for k, v in report.items() if k not in ('accuracy', 'macro avg', 'weighted avg')
])
per_class_f1 = per_class_f1.merge(support_df, on='sub_category', how='left')
per_class_f1 = per_class_f1.sort_values('f1')
per_class_f1.to_csv(f'{P5_DIR}/per_class_f1.csv', index=False)

print('\n  Bottom 15 sub-categories by F1:')
print(per_class_f1.head(15).to_string(index=False))

# Rare classes (fewer than 50 train samples)
rare = per_class_f1[per_class_f1['train_count'] < 50]
print(f'\n  Rare classes (<50 train samples): {len(rare)}')
print(f'  Avg F1 of rare classes: {rare["f1"].mean():.4f}')
print(f'  Avg F1 of non-rare classes: {per_class_f1[per_class_f1["train_count"] >= 50]["f1"].mean():.4f}')

In [ ]:
# ── CELL 10: MILESTONE 5 — Targeted fix: oversample rare + focal loss retrain ─
# Strategy: The main cause of low sub-cat F1 is class imbalance.
# Fix: Oversample rare classes in training data + use Focal Loss.
# Focal Loss down-weights easy (well-classified) examples so the model
# focuses more on hard/rare classes.

print('=' * 60)
print('MILESTONE 5 — Retrain with oversample + Focal Loss')
print('=' * 60)

# ── Focal Loss ────────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight  # class weights tensor

    def forward(self, logits, targets):
        ce   = nn.functional.cross_entropy(logits, targets,
                                           weight=self.weight, reduction='none')
        pt   = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

# ── Oversample rare sub-categories ───────────────────────────────────────────
OVERSAMPLE_THRESHOLD = 100   # classes with fewer than this are oversampled
OVERSAMPLE_TARGET    = 100   # upsample to at least this many

train_aug = train.copy()
for sub_idx in range(num_sub):
    subset = train[train['sub_label'] == sub_idx]
    n = len(subset)
    if 0 < n < OVERSAMPLE_THRESHOLD:
        extra_needed = OVERSAMPLE_TARGET - n
        extras = subset.sample(n=extra_needed, replace=True, random_state=42)
        train_aug = pd.concat([train_aug, extras], ignore_index=True)

train_aug = train_aug.sample(frac=1, random_state=42).reset_index(drop=True)
print(f'  Original train size: {len(train):,}')
print(f'  Augmented train size: {len(train_aug):,}  (+{len(train_aug)-len(train):,} oversampled rows)')

# ── Config ────────────────────────────────────────────────────────────────────
RETRAIN_CONFIG = {
    'model_name':   SUB_CKPT_DIR,   # resume from best Phase 4 checkpoint
    'max_len':      256,
    'epochs':       3,
    'batch_size':   32,
    'lr':           5e-6,            # lower LR since we're fine-tuning further
    'warmup_ratio': 0.10,
    'weight_decay': 0.01,
    'patience':     2,
    'focal_gamma':  2.0,
}
P5_CKPT_DIR = f'{P5_DIR}/scibert_sub_category_v2/best_model'
os.makedirs(P5_CKPT_DIR, exist_ok=True)

# ── Dataloaders ───────────────────────────────────────────────────────────────
tokenizer    = AutoTokenizer.from_pretrained(RETRAIN_CONFIG['model_name'])
num_workers  = 2 if device.type == 'cuda' else 0

train_ds     = PaperDataset(train_aug, tokenizer, RETRAIN_CONFIG['max_len'], 'sub_label')
val_ds       = PaperDataset(val,       tokenizer, RETRAIN_CONFIG['max_len'], 'sub_label')
train_loader = DataLoader(train_ds, batch_size=RETRAIN_CONFIG['batch_size'], shuffle=True,
                          num_workers=num_workers, pin_memory=(device.type=='cuda'),
                          persistent_workers=(num_workers>0))
val_loader   = DataLoader(val_ds,   batch_size=RETRAIN_CONFIG['batch_size']*2, shuffle=False,
                          num_workers=num_workers, pin_memory=(device.type=='cuda'),
                          persistent_workers=(num_workers>0))

# ── Class weights + Focal Loss ────────────────────────────────────────────────
label_counts  = np.bincount(train_aug['sub_label'].values, minlength=num_sub).astype(float)
class_weights = torch.tensor(
    (label_counts.sum() / (num_sub * label_counts)).clip(max=10), dtype=torch.float
).to(device)
loss_fn = FocalLoss(gamma=RETRAIN_CONFIG['focal_gamma'], weight=class_weights)

# ── Model, optimizer, scheduler ──────────────────────────────────────────────
sub_model_v2 = AutoModelForSequenceClassification.from_pretrained(
    RETRAIN_CONFIG['model_name']).to(device)

total_steps  = len(train_loader) * RETRAIN_CONFIG['epochs']
warmup_steps = int(total_steps * RETRAIN_CONFIG['warmup_ratio'])
optimizer    = AdamW(sub_model_v2.parameters(), lr=RETRAIN_CONFIG['lr'],
                     weight_decay=RETRAIN_CONFIG['weight_decay'])
scheduler    = get_linear_schedule_with_warmup(optimizer,
                   num_warmup_steps=warmup_steps, num_training_steps=total_steps)

print(f"  Epochs: {RETRAIN_CONFIG['epochs']}  LR: {RETRAIN_CONFIG['lr']}  "
      f"Focal gamma: {RETRAIN_CONFIG['focal_gamma']}")
print(f'  Total steps: {total_steps:,}  Warmup: {warmup_steps}\n')

# ── Training loop ─────────────────────────────────────────────────────────────
best_val_f1  = 0.0
patience_ctr = 0
log_v2       = []

for epoch in range(1, RETRAIN_CONFIG['epochs'] + 1):
    sub_model_v2.train()
    epoch_loss = 0.0
    t0 = time.time()
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{RETRAIN_CONFIG["epochs"]}', leave=True)
    for step, batch in enumerate(pbar, 1):
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['label'].to(device)
        optimizer.zero_grad()
        out  = sub_model_v2(input_ids=ids, attention_mask=mask)
        loss = loss_fn(out.logits, lbls)
        loss.backward()
        nn.utils.clip_grad_norm_(sub_model_v2.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        epoch_loss += loss.item()
        if step % 10 == 0:
            pbar.set_postfix(loss=f'{epoch_loss/step:.4f}')

    val_loss, val_f1, val_acc = evaluate_loader(sub_model_v2, val_loader, loss_fn)
    elapsed = time.time() - t0
    print(f'  Epoch {epoch}  train_loss={epoch_loss/len(train_loader):.4f}  '
          f'val_loss={val_loss:.4f}  val_macro_f1={val_f1:.4f}  '
          f'val_acc={val_acc:.4f}  time={elapsed:.0f}s')
    log_v2.append({'epoch': epoch, 'val_macro_f1': round(val_f1, 4), 'val_acc': round(val_acc, 4)})

    if val_f1 > best_val_f1:
        best_val_f1  = val_f1
        patience_ctr = 0
        sub_model_v2.save_pretrained(P5_CKPT_DIR)
        tokenizer.save_pretrained(P5_CKPT_DIR)
        print(f'  ✅ New best val macro F1: {best_val_f1:.4f} — checkpoint saved')
    else:
        patience_ctr += 1
        print(f'  No improvement ({patience_ctr}/{RETRAIN_CONFIG["patience"]})')
        if patience_ctr >= RETRAIN_CONFIG['patience']:
            print('  Early stopping.'); break

pd.DataFrame(log_v2).to_csv(f'{P5_DIR}/training_log_v2.csv', index=False)
print(f'\n  Best val macro F1 (v2): {best_val_f1:.4f}')

In [ ]:
# ── CELL 11: MILESTONE 6 — Re-evaluate v2 on full test set ───────────────────
print('=' * 60)
print('MILESTONE 6 — Re-evaluate v2 model on test set')
print('=' * 60)

sub_model_v2_eval = AutoModelForSequenceClassification.from_pretrained(P5_CKPT_DIR).to(device)
tokenizer_v2      = AutoTokenizer.from_pretrained(P5_CKPT_DIR)

test_sub_ds_v2 = PaperDataset(test, tokenizer_v2, RETRAIN_CONFIG['max_len'], 'sub_label')
loader_sub_v2  = DataLoader(test_sub_ds_v2, batch_size=64, shuffle=False,
                             num_workers=num_workers, pin_memory=(device.type=='cuda'))

print('Running v2 sub-category model...')
sub_logits_v2, sub_preds_unc_v2, _ = get_preds_and_logits(sub_model_v2_eval, loader_sub_v2)

# Constrained predictions using Phase 4 category model (unchanged)
sub_preds_con_v2 = np.empty(len(sub_true), dtype=int)
for i, (logit_row, pred_cat) in enumerate(zip(sub_logits_v2, cat_preds)):
    masked = logit_row.copy().astype(float)
    masked[mask_matrix[pred_cat] == 0] = NEG_INF
    sub_preds_con_v2[i] = int(np.argmax(masked))

# Metrics v2
sub_acc_v2   = accuracy_score(sub_true, sub_preds_con_v2)
sub_mf1_v2   = f1_score(sub_true, sub_preds_con_v2, average='macro', zero_division=0)
sub_wf1_v2   = f1_score(sub_true, sub_preds_con_v2, average='weighted', zero_division=0)
joint_acc_v2 = np.mean((cat_preds == cat_true) & (sub_preds_con_v2 == sub_true))
hcons_v2     = np.mean([mask_matrix[c, s] for c, s in zip(cat_preds, sub_preds_con_v2)])

# Comparison table
comparison = pd.DataFrame([
    {'model': 'Phase 4 baseline (CE loss)',      'sub_macro_f1': sub_mf1,    'sub_acc': sub_acc,    'joint_acc': joint_acc},
    {'model': 'Phase 5 v2 (Focal + oversample)', 'sub_macro_f1': sub_mf1_v2, 'sub_acc': sub_acc_v2, 'joint_acc': joint_acc_v2},
    {'model': 'Phase 2 baseline (target)',       'sub_macro_f1': 0.4702,     'sub_acc': '—',        'joint_acc': '—'},
])
print('\n  Results comparison:')
print(comparison.to_string(index=False))

improved = sub_mf1_v2 > sub_mf1
beats_baseline = sub_mf1_v2 > 0.4702
print(f'\n  v2 vs Phase 4: {"✅ improved" if improved else "❌ regressed"} '
      f'({sub_mf1_v2:.4f} vs {sub_mf1:.4f})')
print(f'  v2 vs baseline: {"✅ beats" if beats_baseline else "❌ below"} '
      f'0.4702  (v2={sub_mf1_v2:.4f})')

comparison.to_csv(f'{P5_DIR}/phase5_comparison.csv', index=False)

# Per-class F1 comparison
report_v2 = classification_report(
    sub_true, sub_preds_con_v2,
    target_names=le_sub.classes_, zero_division=0, output_dict=True
)
per_class_v2 = pd.DataFrame([
    {'sub_category': k, 'f1_v2': round(v['f1-score'], 4)}
    for k, v in report_v2.items() if k not in ('accuracy','macro avg','weighted avg')
])
per_class_compare = per_class_f1[['sub_category','f1']].merge(per_class_v2, on='sub_category')
per_class_compare['delta'] = (per_class_compare['f1_v2'] - per_class_compare['f1']).round(4)
per_class_compare = per_class_compare.sort_values('delta')
per_class_compare.to_csv(f'{P5_DIR}/per_class_f1_comparison.csv', index=False)

print('\n  Most improved sub-categories:')
print(per_class_compare.tail(10).to_string(index=False))
print('\n  Most regressed sub-categories:')
print(per_class_compare.head(5).to_string(index=False))

In [ ]:
# ── CELL 12: MILESTONE 7 — Document findings & write report ──────────────────
print('=' * 60)
print('MILESTONE 7 — Phase 5 report')
print('=' * 60)

sep = '=' * 70
report_lines = [
    sep, 'PHASE 5 — ERROR ANALYSIS AND MODEL ITERATION REPORT', sep, '',
    'WHAT WAS TRIED',
    '  Fix 1: Oversampling rare sub-categories (<100 train samples) to 100.',
    '  Fix 2: Replaced CrossEntropy loss with Focal Loss (gamma=2.0).',
    '  Rationale: Root cause analysis showed that most sub-cat errors',
    '  concentrate on rare/overlapping classes. Focal loss penalises',
    '  easy examples less and rare hard examples more.',
    '',
    sep, 'RESULTS COMPARISON', sep,
    comparison.to_string(index=False), '',
    sep, 'PER-CLASS F1 — MOST IMPROVED', sep,
    per_class_compare.tail(10).to_string(index=False), '',
    sep, 'PER-CLASS F1 — MOST REGRESSED', sep,
    per_class_compare.head(5).to_string(index=False), '',
    sep, 'WHAT WORKED', sep,
    f'  Focal loss + oversampling: {"improved" if improved else "did not improve"} macro-F1 '
    f'({sub_mf1:.4f} -> {sub_mf1_v2:.4f})',
    '',
    sep, 'WHAT DID NOT WORK / FUTURE IDEAS', sep,
    '  - Short abstracts remain hard regardless of loss function.',
    '  - Topic overlap between Mathematics/Statistics and Physics/CS',
    '    likely needs label-noise auditing or class merging.',
    '  - Consider multi-task learning (joint cat+sub head) in Phase 6.',
    '',
    sep, 'OUTPUTS', sep,
    f'  {P5_DIR}/cat_errors.csv',
    f'  {P5_DIR}/sub_errors.csv',
    f'  {P5_DIR}/top10_confused_pairs.csv',
    f'  {P5_DIR}/per_class_f1.csv',
    f'  {P5_DIR}/per_class_f1_comparison.csv',
    f'  {P5_DIR}/phase5_comparison.csv',
    f'  {P5_DIR}/confusion_matrix_cat.png',
    f'  {P5_DIR}/scibert_sub_category_v2/best_model  (new checkpoint)',
]
with open(f'{P5_DIR}/phase5_report.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(report_lines))
print(f'  ✓ Report -> {P5_DIR}/phase5_report.txt')

# Update master results table
master_csv = f'{OUTPUT_DIR}/baseline_results.csv'
master = pd.read_csv(master_csv) if os.path.exists(master_csv) else pd.DataFrame()
new_row = pd.DataFrame([{
    'model': 'SciBERT-Hierarchical-v2',
    'task':  'sub_category_constrained',
    'split': 'test',
    'accuracy':    round(sub_acc_v2, 4),
    'macro_f1':    round(sub_mf1_v2, 4),
    'weighted_f1': round(sub_wf1_v2, 4),
    'train_time_s': '—',
}])
master = pd.concat([master, new_row], ignore_index=True)
master.to_csv(master_csv, index=False)
print(f'  ✓ Master results updated -> {master_csv}')

print('\n' + sep)
print('PHASE 5 COMPLETE')
print(sep)
print(f'  Category:          acc={cat_acc:.4f}  macro-F1={cat_mf1:.4f} (unchanged)')
print(f'  Sub-cat v1:        acc={sub_acc:.4f}  macro-F1={sub_mf1:.4f}')
print(f'  Sub-cat v2:        acc={sub_acc_v2:.4f}  macro-F1={sub_mf1_v2:.4f}')
print(f'  Joint acc v2:      {joint_acc_v2:.4f}')
print(f'  H-consistency v2:  {hcons_v2:.4f}')

In [ ]:
# ── CELL 13: Save everything back to Google Drive ────────────────────────────
import shutil
DRIVE_SAVE = '/content/drive/MyDrive/PaperChaseAI-outputs'
if os.path.exists(DRIVE_SAVE):
    shutil.rmtree(DRIVE_SAVE)
shutil.copytree(f'{EXTRACT_DIR}/outputs', DRIVE_SAVE)
print(f'✅ All outputs saved to Drive: {DRIVE_SAVE}')